**CAPE Pretraining**

In [1]:
#!/usr/bin/env python
# coding: utf-8

import sys
import os
import h5py
import glob
import csv
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import utils.utils as u
from utils.build_model import *

import tensorflow as tf
import keras
from tensorflow.keras.models import load_model, Model
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LearningRateScheduler

np.random.seed(10)

2025-07-18 16:43:39.036041: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-18 16:43:39.082418: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-18 16:43:39.083079: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-18 16:43:39.869262: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [15]:
"""
Trainig Configuraion
"""
class pretraining:
    # params
    def __init__(self, num_leads = 8, num_samples = 2800, num_feats = 256, 
                 tmask = 0.2, mode = 'test'):
        """
        Trainig Configuraion
        """
        self.num_leads = num_leads
        self.num_samples = num_samples #2800 = 7 sec
        self.num_feats = num_feats
        self.tmask = tmask
        if self.tmask: self.tmask = int(self.tmask * self.num_samples)
            
        self.lead_id = 0 # if one lead
        self.transfer = 0 # starting from pretrained network
        self.load = 1 # load all data
        self.datasets = ['mimic'] # training datasets
        self.lr = 0.01 if self.transfer else 0.1
        self.ecg_model = Pretrain_model(num_channels=self.num_leads, num_samples=self.num_samples, num_feats=self.num_feats)
        """
        Transfer learning params
        """
        self.initial_weights = 'models/CAPE.h5'

        """
        Small 'test' configurations vs. full 'train' configurations
        """
        self.mode = mode
        self.epochs = 1 if mode == 'test' else 200
        self.half_batch_size = 32 if mode == 'test' else 512
        self.data_size = 100000 if mode == 'test' else 0
        self.num={}
        self.num['train'] = 1000 if mode == 'test' else 0
        self.num['val'] = 1000 if mode == 'test' else 0
        self.sl = 100 if mode == 'test' else 10000
        self.resume = 0

        """
        Training output path. Resume if checkpoints exist
        """
        self.train_dir = 'models'
        out_dir = f'CAPE_{num_leads}leads_{num_feats}feats_{num_samples}samples_{str(self.lr).replace(".", "p")}lr_{self.epochs}'
        if self.transfer: out_dir = out_dir + '_transfer'
        if self.tmask: out_dir = out_dir + f'_{tmask}tmask'
        for dataset in self.datasets:
            out_dir = out_dir + f'_{dataset}'
        if self.mode == 'test': out_dir = out_dir + '_test'
        self.res_dir = os.path.join(self.train_dir, out_dir)
        
        self.remote_path = '/rds/general/project/fsn_ai_ecg_code/live/ECG Datasets/' 
        self.local_path = 'data/ECGdata/' # save data here
        self.data_key = 'ECGs'
        
    def get_indexes(self, data_info):
        """
        Load indexes and data
        """
        data_info['index'] = {}
        data_info['len'] = {}
        data_info['data'] = {}
        data_info['shape'] = (self.num_samples, self.num_leads)

        indexes ={}
        indexes['train'] = {}
        indexes['val'] = {}
        data_info['index']['train'] = {}
        data_info['index']['val'] = {}
        hf_data ={}
        data_dir ={}
        data = {}
        data_len = 0
        total_index = {}
        for i, dataset in enumerate(self.datasets):
           data_dir[dataset] = os.path.join(self.remote_path, data_info['path'][dataset]) if os.path.isdir(self.remote_path) else os.path.join(self.local_path, data_info['path'][dataset])
           data[dataset] = h5py.File(os.path.join(data_dir[dataset], 'ecg_data.h5'), 'r')[self.data_key]
           print(f'data for {self.data_key} at {data_dir[dataset]} .....')
           data_info['len'][dataset] = min(len(data[dataset]), self.data_size) if self.data_size else len(data[dataset])
           ind_dir = os.path.join('indexes', dataset, f'indexes_{dataset}')
           #data_info['index'][dataset] = {}
           data_info['data'][dataset] = data_len
           data_len += data_info['len'][dataset]
           if self.load:
             init = time.time()
             data[dataset] = np.array(data[dataset]) if not self.data_size else np.array(data[dataset][:self.data_size])
             print(f'data of len {len(data[dataset])} => {data[dataset].shape} in {time.time()-init:0.4f} s')
           for split in ['train', 'val']:
             if i==0: total_index[split] =  0

             init = time.time()
             ind =  u.load_indexes(ind_dir, name=split, epochs=self.epochs)
             if self.data_size:
                ind_data = np.all(ind<self.data_size, axis=(1, 2))
                ind= ind[ind_data]
                print(f'{split} indexes of shape = {ind.shape} min={np.min(ind)} max={np.max(ind)}')
             if self.num[split]: ind = ind[:self.num[split]]
             indexes[split][dataset] = ind
             data_info['index'][split][dataset] = [total_index[split]]
             total_index[split] += len(ind)
             data_info['index'][split][dataset].append(total_index[split])      
        return data_info, data, indexes, total_index

    def shuffle(self, arr):
        np.random.shuffle(arr)
        return arr

    def get_dataset(self, ind, ind_offset):
        for dataset in ind_offset:
            #print(ind, ind_offset[dataset][0], ind_offset[dataset][1])
            if (ind >= ind_offset[dataset][0]) and (ind < ind_offset[dataset][1]):
                return dataset, ind_offset[dataset][0]

    def get_ecg_batch(self, batch_ind, indexes, ind_offset, data, epoch, data_info, verbose=0):
       if verbose: init = time.time()
       first_half_batch = np.zeros((len(batch_ind), self.num_samples, self.num_leads))
       second_half_batch = np.zeros((len(batch_ind), self.num_samples, self.num_leads))
       #print('epoch', epoch)
       for n in range(len(batch_ind)):
           dataset, offset = self.get_dataset(batch_ind[n], ind_offset)
           index = indexes[dataset][batch_ind[n] - offset]
           
           start = np.random.randint(100, data_info['shape'][dataset][1]-self.num_samples-100, size=2) if data_info['shape'][dataset][1] > self.num_samples else [0, 0]
           ecg1 = np.array(data[dataset][index[epoch, 0]])[start[0]:start[0] + self.num_samples, :]
           ecg2 = np.array(data[dataset][index[epoch, 1]])[start[1]:start[1] + self.num_samples, :]
           if self.tmask:
                start = np.random.randint(len(ecg1), size=2)
                ecg1[start[0]: start[0] + self.tmask]=0
                ecg2[start[1]: start[1] + self.tmask]=0

           first_half_batch[n] = ecg1
           second_half_batch[n] = ecg2

       ecg_batch = np.concatenate((first_half_batch, second_half_batch), axis=0)
       if self.num_leads == 12: ecg_batch = u.process_ecg_8lead(ecg_batch)
       elif self.num_leads == 1: ecg_batch = ecg_batch[:, :, self.lead_id]
       if verbose: print(f'batch with {ecg_batch.shape} is generated in {time.time()-init} seconds')
       return ecg_batch

    def batch_generator(self, total_ind, indexes, ind_offset, data, data_info, start_epoch=-1):
        batch_size = 2 * self.half_batch_size
        epoch = start_epoch
        while True:
          ind = np.random.permutation(total_ind)
          epoch+=1
          print(f'starting epoch {epoch}')
          for batch_idx in range(0, len(ind), self.half_batch_size):
             #print(batch_idx)
             batch_ids = ind[batch_idx : batch_idx+self.half_batch_size]
             #batch_ids = indexes[ind(batch_idx : batch_idx+half_batch_size]
             verbose = 1 if batch_idx==0 else 0
             ecg_batch = self.get_ecg_batch(batch_ids, indexes, ind_offset, data, epoch, data_info)
             yield ecg_batch, np.zeros((ecg_batch.shape[0],))



#pre.ecg_model.summary()

**Pretraining initialization**

Check if training was previously aborted and resume from the last checkpoint if available. Otherwise, start a new training session. If using transfer learning, start from a pre-trained model.

In [16]:
pre = pretraining()

if not os.path.isdir(pre.res_dir):
  u.safe_mkdir(pre.res_dir)
  print(f'creating directory {pre.res_dir} for results')
else:
  checkpoints = sorted(glob.glob(os.path.join(pre.res_dir, 'checkpoint*.h5')))
  if len(checkpoints) > 0:
    pre.ecg_model.load_weights(checkpoints[-1])
    history1 = pd.read_csv(os.path.join(pre.res_dir, 'model_history_log.csv'))
    lr1 = np.array(history1.lr)[-1]
    resume_epoch = np.array(history1.epoch)[-1]
    print(f'loaded weights for epoch{resume_epoch} from {checkpoints[-1]}')
    resume = 1

if pre.transfer:
    by_name = True if 'h5' in pre.initial_weights else False
    pre.ecg_model.load_weights(pre.initial_weights, by_name=by_name)

**Read Indexes**

for two randomly selected ecgs from same patient for each epoch. shape = (num_samples, epochs, 2)

In [4]:
data_info = u.get_data_info()
data_info, data, indexes, total_index = pre.get_indexes(data_info)

data for ECGs at /rds/general/project/fsn_ai_ecg_code/live/ECG Datasets/mimic_braveheart_rhythm_8l_400_bf .....
data of len 100000 => (100000, 4096, 8) in 5.5228 s
loading indexes from indexes/mimic/indexes_mimic/indexes_train.csv
indexes of shape (92655, 400, 2) are loaded for 400 epochs
indexes of shape (92655, 1, 2) are returned
train indexes of shape = (1525, 1, 2) min=9 max=99997
loading indexes from indexes/mimic/indexes_mimic/indexes_val.csv
indexes of shape (10295, 400, 2) are loaded for 400 epochs
indexes of shape (10295, 1, 2) are returned
val indexes of shape = (186, 1, 2) min=283 max=99961


The data shape for each pretraining dataset.

In [6]:
data_info['shape']={}
for dataset in pre.datasets:
  data_info['shape'][dataset]= data[dataset].shape[1:]

**Data generators**

In [17]:
start_epoch = resume_epoch if pre.resume else -1
train_gen = pre.batch_generator(total_index['train'], indexes['train'], data_info['index']['train'], data, data_info, start_epoch=start_epoch)
val_gen = pre.batch_generator(total_index['val'], indexes['val'], data_info['index']['val'], data, data_info, start_epoch=start_epoch)

**Testing data generators**

In [18]:
"""
num_batches=5
init = time.time()

for n in range(num_batches):
    try:
        batch = next(train_gen)
        print('train', batch[0].shape)
        batch = next(val_gen)
        print('val', batch[0].shape)
    except:
        print('end')
end_time = time.time()-init
print(f' {num_batches} batches of shape {batch[0].shape} generated in {end_time} s')
print(f'{(end_time/num_batches)} s / batch')
print(f'{(end_time)/(2 * pre.half_batch_size * num_batches)} s / sample')
"""

"\nnum_batches=5\ninit = time.time()\n\nfor n in range(num_batches):\n    try:\n        batch = next(train_gen)\n        print('train', batch[0].shape)\n        batch = next(val_gen)\n        print('val', batch[0].shape)\n    except:\n        print('end')\nend_time = time.time()-init\nprint(f' {num_batches} batches of shape {batch[0].shape} generated in {end_time} s')\nprint(f'{(end_time/num_batches)} s / batch')\nprint(f'{(end_time)/(2 * pre.half_batch_size * num_batches)} s / sample')\n"

In [19]:
u.safe_mkdir(pre.res_dir)
print(f'creating directory {pre.res_dir} for results')
if pre.transfer: pre.ecg_model.load_weights('models/CAPE_mimic', by_name=True)
#pre.ecg_model.summary()
model_checkpoint_callback = ModelCheckpoint(filepath=os.path.join(pre.res_dir, 'best_model'),
                                monitor='val_loss',
                                mode='min',
                                save_best_only=True)
csv_logger = tf.keras.callbacks.CSVLogger( os.path.join(pre.res_dir,"model_history_log.csv"), append=True)
checkpoint_path = os.path.join(pre.res_dir,'checkpoint_{epoch:04d}.hdf5')
cp_callback = tf.keras.callbacks.ModelCheckpoint(
   checkpoint_path, verbose=1, save_weights_only=True,
   # Save weights, every epoch.
   save_freq='epoch')
lr_decayed_fn = tf.keras.optimizers.schedules.CosineDecay(pre.lr, pre.epochs)
lr_decay = LearningRateScheduler(lr_decayed_fn)
init = time.time()
train_steps = (total_index['train']//pre.half_batch_size) #np.ceil(len(train_indexes)/half_batch_size)
val_steps = (total_index['val']//pre.half_batch_size) #np.ceil(len(val_indexes)/half_batch_size)

creating directory models/CAPE_8leads_256feats_2800samples_0p1lr_1_0.2tmask_mimic_test for results


In [20]:
history = pre.ecg_model.fit(
        x=train_gen,
        steps_per_epoch=train_steps,
        validation_data=val_gen,
        validation_steps=val_steps,
        batch_size = pre.half_batch_size,
        epochs=pre.epochs,
        verbose=1,
        callbacks=[cp_callback, lr_decay, csv_logger, model_checkpoint_callback])
train_size = total_index['train']
print(print(f'{pre.epochs} epochs took {time.time()-init} seconds for {train_size} ids'))



starting epoch 0
epoch 0


2025-07-18 17:05:39.739229: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


epoch 0
31/31 [==============================] - ETA: 0s - loss: 7.4007 - simclr_accuracy: 0.1094 starting epoch 0
epoch 0


2025-07-18 17:15:21.918153: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


epoch 0
epoch 0
epoch 0
epoch 0
epoch 0

Epoch 1: saving model to models/CAPE_8leads_256feats_2800samples_0p1lr_1_0.2tmask_mimic_test/checkpoint_0001.hdf5


INFO:tensorflow:Assets written to: models/CAPE_8leads_256feats_2800samples_0p1lr_1_0.2tmask_mimic_test/best_model/assets


INFO:tensorflow:Assets written to: models/CAPE_8leads_256feats_2800samples_0p1lr_1_0.2tmask_mimic_test/best_model/assets


31/31 [==============================] - 606s 19s/step - loss: 7.4007 - simclr_accuracy: 0.1094 - val_loss: 8.0908 - val_simclr_accuracy: 0.1031 - lr: 0.1000
1 epochs took 721.2791647911072 seconds for 1000 ids
None


In [ ]:
history_dict = history.history
u.plot_loss(history_dict['loss'], history_dict['val_loss'], 'loss', pre.res_dir)
u.plot_loss(history_dict['simclr_accuracy'], history_dict['val_simclr_accuracy'], 'simclr_accuracy', pre.res_dir)


In [ ]:
history.history